# Classes: Building Wordle Players

We already have a `score` function.  Today we build a Wordle game that different
players can play: a random bot, a smarter bot, and an LLM.  Along the way we will
see why **classes** are useful.

Run each cell, but try to **predict the output before you run it**.

## Setup

The word lists are text files in the course GitHub repository, one word per line.
`urllib.request.urlopen` opens a file on the web the way `open` opens a file on
your computer, so this works on your own computer and on Colab.  `.decode()` turns
the downloaded bytes into a string.

In [ ]:
import random
import urllib.request

URL = "https://raw.githubusercontent.com/hcai-programming/hcai121-fall2026/main/week10/"

def load_words(filename):
    with urllib.request.urlopen(URL + filename) as f:
        return f.read().decode().split()

answers = load_words("answers.txt")   # words that can be the secret word
guesses = load_words("valid_guesses.txt")   # every legal guess
legal = set(guesses)
print(len(answers), "possible answers,", len(guesses), "legal guesses")

In [ ]:
def score(answer, guess):
    """Return a list of (letter, status) pairs, where status is  green,  yellow, orgray."""
    status = ["gray"] * len(guess)
    remaining = {}
    for i, (g, a) in enumerate(zip(guess, answer)):
        if g == a:
            status[i] = "green"
        else:
            remaining[a] = remaining.get(a, 0) + 1
    for i, (g, a) in enumerate(zip(guess, answer)):
        if status[i] == "gray" and remaining.get(g, 0) > 0:
            status[i] = "yellow"
            remaining[g] = remaining[g] - 1
    return list(zip(guess, status))

print(score("crane", "acorn"))

## 1. A game that works with any strategy

A *strategy* is a function that looks at the guesses so far (`history`) and
returns the next guess.  The game loop does not care which strategy it gets: we
pass the function in as an argument, just like `key=` in `sorted`.

In [ ]:
def random_guess(history):
    return random.choice(guesses)

def play(answer, choose_guess, max_turns=6):
    history = []
    for turn in range(1, max_turns + 1):
        guess = choose_guess(history)
        feedback = score(answer, guess)
        history.append((guess, feedback))
        if guess == answer:
            return turn
    return None

print(play("crane", random_guess))

`None` means it did not solve the puzzle in 6 turns.  Guessing at random almost never works.

## 2. A smarter strategy needs a memory

A better idea: keep a list of the words that are still possible, and after every
guess throw out the words that do not match the feedback.  The strategy has to
**remember** that list from one turn to the next, so we keep it in a variable
outside the function.  (`global` lets a function change a variable that was
created outside it.)

In [ ]:
candidates = list(answers)

def filter_guess(history):
    global candidates
    if history:
        last_guess, last_feedback = history[-1]
        candidates = [w for w in candidates
                      if score(w, last_guess) == last_feedback]
    return candidates[0]

print(play("crane", filter_guess))
print("still possible at the end:", candidates)

It solved the puzzle in 4 turns.  Now play a second game:

In [ ]:
print(play("slate", filter_guess))

It crashed.  The list still held the leftovers from the first game (`crane`,
`crate`, `crave`, `craze`).  The second game guessed `crane`, threw out every word
on the list after the first clue, and then asked for `candidates[0]` from an empty
list.

We could remember to reset `candidates` before every game.  But what if we want
two of these players, each with its own list, in the same program?  One global
variable cannot hold two memories.

## 3. Classes: bundling memory with behavior

A **class** is a blueprint.  An **object** made from the class has its own data
(its *attributes*) and the functions that work on that data (its *methods*).

- `__init__` runs when an object is created.  It sets the object's attributes.
- `self` is the object itself: `self.candidates` means *this* player's list.
- A method is a function defined inside the class.  Call it with a dot:
  `player.next_guess(history)`.

In [ ]:
class FilterPlayer:
    def __init__(self, name):
        self.name = name
        self.candidates = []

    def new_game(self):
        self.candidates = list(answers)

    def next_guess(self, history):
        if history:
            last_guess, last_feedback = history[-1]
            self.candidates = [w for w in self.candidates
                               if score(w, last_guess) == last_feedback]
        return self.candidates[0]

The game now asks a *player object* for each guess, and it calls
`new_game()` first, so every game starts fresh.

In [ ]:
def play(answer, player, max_turns=6):
    player.new_game()
    history = []
    for turn in range(1, max_turns + 1):
        guess = player.next_guess(history)
        feedback = score(answer, guess)
        history.append((guess, feedback))
        if guess == answer:
            return turn
    return None

In [ ]:
p1 = FilterPlayer("Ana's bot")
p2 = FilterPlayer("Ben's bot")

print(play("crane", p1))
print(play("slate", p1))          # the game that crashed before
print(play("ghost", p2))

print(p1.name, len(p1.candidates))
print(p2.name, len(p2.candidates))

Two objects from the same class, each with its own name and its own list.
Neither one can mess up the other.

## 4. One interface, many players

Any object with a `name`, a `new_game()` method, and a `next_guess(history)`
method can play: `play` does not care what kind of player it gets.  Agreeing on
this **interface** first is a basic idea of object-oriented design.  It lets us
add new kinds of players without changing the game.

In [ ]:
class RandomPlayer:
    def __init__(self, name):
        self.name = name

    def new_game(self):
        pass                          # nothing to reset

    def next_guess(self, history):
        return random.choice(guesses)

print(play("crane", RandomPlayer("random")))

The LLM player needs its own memory too: which model to use, and how many
illegal guesses it has made.  The game rules go in the **system** message, and
the **user** message is the feedback so far.  (This needs Ollama running, as in
the last notebook.)

In [ ]:
import ollama

RULES = ("You are playing Wordle. The answer is a 5-letter English word. "
         "After each guess you get feedback for each letter: green means the "
         "right letter in the right spot, yellow means the letter is in the "
         "word but in a different spot, gray means the letter is not in the "
         "word. Reply with only your next guess: one 5-letter word.")

class LLMPlayer:
    def __init__(self, name, model):
        self.name = name
        self.model = model
        self.illegal = 0              # how many times it broke the rules

    def new_game(self):
        pass

    def next_guess(self, history):
        lines = []
        for guess, feedback in history:
            letters = [f"{letter.upper()}={status}" for letter, status in feedback]
            lines.append(guess.upper() + ": " + " ".join(letters))
        if not lines:
            lines = ["(no guesses yet)"]
        messages = [
            {"role": "system", "content": RULES},
            {"role": "user", "content": "Guesses so far:\n" + "\n".join(lines)},
        ]
        reply = ollama.chat(model=self.model, messages=messages)
        word = reply["message"]["content"].strip().strip(".!").lower()
        if word in legal:
            return word
        self.illegal = self.illegal + 1
        return random.choice(guesses)   # an illegal reply costs a random guess

In [ ]:
llm = LLMPlayer("llama", "llama3.2")
print(play("crane", llm))
print("illegal guesses so far:", llm.illegal)

## 5. A tournament

Because every player has the same interface, a tournament is a loop over a list
of player objects.  We save every result in a CSV file.

In [ ]:
import csv

random.seed(1)
secret_words = random.sample(answers, 20)

def tournament(players, secret_words, filename):
    with open(filename, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["player", "answer", "turns"])
        for player in players:
            for answer in secret_words:
                writer.writerow([player.name, answer, play(answer, player)])

players = [RandomPlayer("random"), FilterPlayer("filter")]
tournament(players, secret_words, "results.csv")
print("saved results.csv")

Read the results back and make a leaderboard (a blank `turns` means the player lost):

In [ ]:
wins = {}
total_turns = {}
with open("results.csv") as f:
    for row in csv.DictReader(f):
        name = row["player"]
        wins[name] = wins.get(name, 0)
        total_turns[name] = total_turns.get(name, 0)
        if row["turns"] != "":
            wins[name] = wins[name] + 1
            total_turns[name] = total_turns[name] + int(row["turns"])

for name in wins:
    if wins[name] > 0:
        average = round(total_turns[name] / wins[name], 2)
    else:
        average = None
    print(f"{name}: won {wins[name]} of {len(secret_words)}, average turns when winning: {average}")

Adding the LLM to the tournament is one more object in the list; nothing
else changes.  It is slow, so we only use 5 words.

In [ ]:
players = [RandomPlayer("random"), FilterPlayer("filter"), LLMPlayer("llama", "llama3.2")]
tournament(players, secret_words[:5], "results_llm.csv")

with open("results_llm.csv") as f:
    print(f.read())